In [ ]:
# Cell 0 — Bootstrap / Installs
# Expected output: package checks, environment banner, mission + release notes table.

import importlib
import json
import math
import os
import random
import shutil
import subprocess
import sys
import time
import zipfile
from dataclasses import asdict, dataclass
from pathlib import Path
from textwrap import dedent

from IPython.display import HTML, Markdown, display

TITLE = "# APTOS-DR-GPU v4 — GPU-first, portfolio-ready"
MISSION = "GPU-first diabetic retinopathy training pipeline with reproducible results, polished artifacts, and clean Git history."
DISCLAIMER = "**Clinical Disclaimer:** Research use only — not for clinical diagnosis."

release_notes = """
## v4 Release Notes
- Bumped notebook version branding to v4.
- Cleaned up AMP/autocast usage to remove deprecation warnings.
- Replaced the warning-prone training augmentation with a supported affine transform.
- Added automatic 7z installation in the bootstrap so extraction uses the fast path.
- Added class-balanced sampling and weighted loss so training better reflects severity imbalance.
- Added richer validation metrics beyond plain accuracy/QWK.
- Kept the fast local-extraction path and H100 defaults intact.
- Preserved gradient accumulation, temperature scaling, TTA, and OOF reporting.
- Notebook outputs were cleared for a clean source-only Git commit.
- All cells remain idempotent and safe to rerun.
"""

quick_start = """
## Quick Start (RunPod / Local)
| Cell | Purpose | Typical time (first run) |
|---|---|---|
| 0 | Bootstrap / installs | 1–4 min |
| 1 | Environment + config | < 1 min |
| 2 | Dataset verification + fast extraction | 3–10 min |
| 3 | Data exploration + splitting | 1–3 min |
| 4 | Dataset + dataloaders | 1–2 min |
| 5 | Model + loss + mixup | < 1 min |
| 6 | Training loop | 15–120 min |
| 7 | Temperature scaling + TTA | 2–10 min |
| 8 | Evaluation + OOF | 1–5 min |
| 9 | Visualization + artifacts | 1–3 min |
| 10 | Final summary + checklist | < 1 min |

## Portfolio Highlights
- **GPU-first:** BF16/TF32, `channels_last`, `torch.compile`, pinned memory, and non-blocking transfers.
- **Severity-aware:** class-balanced sampling, weighted loss, balanced accuracy, macro F1, and per-class recall.
- **Reproducible:** fixed seeds, config capture, checkpointing, and versioned artifacts.
- **Source-clean:** notebook outputs removed before push for a professional Git diff.

## Recommended RunPod template
`runpod/pytorch:2.8.0-py3.11-cuda12.8.1-cudnn-devel-ubuntu22.04`
"""

NOTEBOOK_UI = """
<style>
  .aptos-hero {
    background: linear-gradient(135deg, #0f172a 0%, #111827 45%, #1e293b 100%);
    color: #e5eefb;
    border: 1px solid rgba(148, 163, 184, 0.18);
    border-radius: 18px;
    padding: 22px 24px;
    box-shadow: 0 10px 30px rgba(15, 23, 42, 0.18);
    margin: 8px 0 16px 0;
  }
  .aptos-hero h1 {
    margin: 0 0 6px 0;
    font-size: 30px;
    line-height: 1.1;
    letter-spacing: 0.2px;
  }
  .aptos-hero p {
    margin: 6px 0 0 0;
    font-size: 14px;
    color: #cbd5e1;
  }
  .aptos-badges {
    display: flex;
    flex-wrap: wrap;
    gap: 8px;
    margin-top: 14px;
  }
  .aptos-badge {
    background: rgba(59, 130, 246, 0.16);
    color: #dbeafe;
    border: 1px solid rgba(96, 165, 250, 0.35);
    border-radius: 999px;
    padding: 6px 10px;
    font-size: 12px;
    font-weight: 600;
  }
  .aptos-grid {
    display: grid;
    grid-template-columns: repeat(auto-fit, minmax(180px, 1fr));
    gap: 12px;
    margin: 12px 0 18px 0;
  }
  .aptos-card {
    background: #ffffff;
    border: 1px solid #e5e7eb;
    border-radius: 16px;
    padding: 14px 16px;
    box-shadow: 0 4px 12px rgba(15, 23, 42, 0.06);
  }
  .aptos-card .label {
    font-size: 12px;
    text-transform: uppercase;
    letter-spacing: 0.08em;
    color: #64748b;
    margin-bottom: 6px;
    font-weight: 700;
  }
  .aptos-card .value {
    font-size: 18px;
    font-weight: 800;
    color: #0f172a;
  }
  .aptos-card .sub {
    font-size: 12px;
    color: #475569;
    margin-top: 4px;
  }
</style>
"""

display(HTML(NOTEBOOK_UI))
display(HTML(f"""
<div class="aptos-hero">
  <h1>APTOS-DR-GPU v4</h1>
  <p>{MISSION}</p>
  <div class="aptos-badges">
    <span class="aptos-badge">GPU-first training</span>
    <span class="aptos-badge">TF32 + bf16 ready</span>
    <span class="aptos-badge">channels_last enabled</span>
    <span class="aptos-badge">compile + inference_mode</span>
    <span class="aptos-badge">clinical severity aware</span>
  </div>
</div>
"""))
display(Markdown(DISCLAIMER))
display(Markdown(release_notes))
display(Markdown(quick_start))

display(Markdown("""
## Recruiter Snapshot
This notebook is designed to read like a strong portfolio artifact:
- modern GPU utilization with practical training throughput
- clinical-severity-aware metrics instead of shallow accuracy-only reporting
- reproducible runs with versioned outputs and clean experiment tracking in the code itself
- polished presentation that is ready to share in Git or a demo
"""))

REQUIRED_PACKAGES = {
    "numpy": "numpy",
    "pandas": "pandas",
    "scikit-learn": "scikit-learn",
    "matplotlib": "matplotlib",
    "seaborn": "seaborn",
    "Pillow": "Pillow",
    "tqdm": "tqdm",
    "opencv-python-headless": "opencv-python-headless",
    "timm": "timm",
    "torchvision": "torchvision",
}


def _is_importable(module_name: str) -> bool:
    try:
        return importlib.util.find_spec(module_name) is not None
    except Exception:
        return False


def ensure_python_packages() -> None:
    missing = []
    module_probe_map = {
        "numpy": "numpy",
        "pandas": "pandas",
        "scikit-learn": "sklearn",
        "matplotlib": "matplotlib",
        "seaborn": "seaborn",
        "Pillow": "PIL",
        "tqdm": "tqdm",
        "opencv-python-headless": "cv2",
        "timm": "timm",
        "torchvision": "torchvision",
    }
    for pkg, probe in module_probe_map.items():
        if not _is_importable(probe):
            missing.append(REQUIRED_PACKAGES[pkg])

    if not missing:
        print("✅ Python dependencies already satisfied")
        return

    print(f"Installing missing packages: {missing}")
    cmd = [sys.executable, "-m", "pip", "install", "-q"] + missing
    subprocess.run(cmd, check=True)
    print("✅ Package installation complete")


ensure_python_packages()

import cv2
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
import timm
import torch
import torch.nn as nn
import torch.nn.functional as F
import torchvision.transforms as T
from PIL import Image
from sklearn.metrics import accuracy_score, balanced_accuracy_score, cohen_kappa_score, confusion_matrix, f1_score, recall_score
from sklearn.model_selection import StratifiedKFold
from torch.utils.data import DataLoader, Dataset, WeightedRandomSampler
from tqdm.auto import tqdm

NOTEBOOK_VERSION = "v4"
RESEARCH_DISCLAIMER = "Research use only — not for clinical diagnosis."

print(f"Python: {sys.version.split()[0]}")
print(f"Platform: {sys.platform}")
print("Cell 0 complete")

In [ ]:
# Cell 1 — Environment + Config (central GPU hub)
# Expected output: hardware summary, vRAM table, deterministic setup, resolved config.

# Recommended RunPod template: `runpod/pytorch:2.8.0-py3.11-cuda12.8.1-cudnn-devel-ubuntu22.04`
display(Markdown("**Recommended RunPod template:** `runpod/pytorch:2.8.0-py3.11-cuda12.8.1-cudnn-devel-ubuntu22.04`"))

NOTEBOOK_VERSION = "v4"
RESEARCH_DISCLAIMER = "Research use only — not for clinical diagnosis."


def seed_everything(seed: int = 42, deterministic: bool = False) -> None:
    random.seed(seed)
    np.random.seed(seed)
    os.environ["PYTHONHASHSEED"] = str(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    if torch.cuda.is_available():
        torch.backends.cudnn.deterministic = deterministic
        torch.backends.cudnn.benchmark = not deterministic


@dataclass
class TrainConfig:
    seed: int = int(os.environ.get("SEED", "42"))
    project_name: str = os.environ.get("PROJECT_NAME", "APTOS-DR-GPU")
    run_name: str = os.environ.get("RUN_NAME", f"{NOTEBOOK_VERSION}_convnextv2_tiny")
    backbone: str = os.environ.get("BACKBONE", "convnextv2_tiny")

    img_size: int = int(os.environ.get("IMG_SIZE", "384"))
    batch_size: int = int(os.environ.get("BATCH_SIZE", "36"))
    effective_batch_target: int = int(os.environ.get("EFFECTIVE_BATCH_TARGET", "192"))
    epochs: int = int(os.environ.get("EPOCHS", "12"))
    lr: float = float(os.environ.get("LR", "3e-4"))
    weight_decay: float = float(os.environ.get("WEIGHT_DECAY", "1e-4"))
    num_workers: int = int(os.environ.get("NUM_WORKERS", str(min(24, max(8, (os.cpu_count() or 8) - 2)))))

    fold: int = int(os.environ.get("FOLD", "0"))
    n_folds: int = int(os.environ.get("N_FOLDS", "5"))
    amp: bool = os.environ.get("USE_AMP", "1") == "1"
    use_torch_compile: bool = os.environ.get("USE_TORCH_COMPILE", "1") == "1"
    tta: int = int(os.environ.get("TTA", "4"))

    label_smoothing: float = float(os.environ.get("LABEL_SMOOTHING", "0.05"))
    mixup_alpha: float = float(os.environ.get("MIXUP_ALPHA", "0.2"))
    cutmix_alpha: float = float(os.environ.get("CUTMIX_ALPHA", "1.0"))
    mix_prob: float = float(os.environ.get("MIX_PROB", "1.0"))

    grad_clip: float = float(os.environ.get("GRAD_CLIP", "1.0"))
    patience: int = int(os.environ.get("PATIENCE", "4"))

    deterministic: bool = os.environ.get("DETERMINISTIC", "0") == "1"


CFG = TrainConfig()
seed_everything(CFG.seed, deterministic=CFG.deterministic)

CFG.grad_accum_steps = max(1, math.ceil(CFG.effective_batch_target / max(1, CFG.batch_size)))
CFG.effective_batch_size = CFG.grad_accum_steps * CFG.batch_size

ROOT = Path.cwd()
RESULTS_DIR = Path(os.environ.get("RESULTS_DIR", "results"))
ARTIFACTS_DIR = Path(os.environ.get("ARTIFACTS_DIR", "artifacts"))
CHECKPOINTS_DIR = Path(os.environ.get("CHECKPOINTS_DIR", "checkpoints"))

for p in [RESULTS_DIR, ARTIFACTS_DIR, CHECKPOINTS_DIR]:
    p.mkdir(parents=True, exist_ok=True)

if torch.cuda.is_available():
    DEVICE = torch.device("cuda")
    gpu_name = torch.cuda.get_device_name(0)
    total_vram_gb = torch.cuda.get_device_properties(0).total_memory / (1024**3)
    torch.backends.cuda.matmul.allow_tf32 = True
    torch.backends.cudnn.allow_tf32 = True
    torch.set_float32_matmul_precision("high")
    AMP_DTYPE = torch.bfloat16 if torch.cuda.get_device_capability(0)[0] >= 8 else torch.float16
    if not CFG.deterministic:
        torch.backends.cudnn.benchmark = True
else:
    DEVICE = torch.device("cpu")
    gpu_name = "NO CUDA DEVICE"
    total_vram_gb = 0.0
    AMP_DTYPE = torch.float32

# GPU-aware defaults for high-VRAM cards.
if DEVICE.type == "cuda" and total_vram_gb >= 70 and CFG.img_size >= 384:
    CFG.batch_size = max(CFG.batch_size, 64)
    CFG.effective_batch_target = max(CFG.effective_batch_target, 384)
    CFG.num_workers = max(CFG.num_workers, min(32, max(12, (os.cpu_count() or 8) - 2)))
    CFG.grad_accum_steps = max(1, math.ceil(CFG.effective_batch_target / max(1, CFG.batch_size)))
    CFG.effective_batch_size = CFG.grad_accum_steps * CFG.batch_size

print(f"Notebook version: {NOTEBOOK_VERSION}")
print(RESEARCH_DISCLAIMER)
print(f"PyTorch: {torch.__version__}")
print(f"CUDA runtime: {torch.version.cuda}")
print(f"Device: {DEVICE} | {gpu_name}")
print(f"Total VRAM: {total_vram_gb:.1f} GB")
print(f"AMP dtype: {AMP_DTYPE}")
print(f"Deterministic: {CFG.deterministic}")
print(f"cuDNN benchmark: {torch.backends.cudnn.benchmark if torch.cuda.is_available() else False}")

if not str(torch.__version__).startswith("2.8"):
    print("⚠️ Recommended: PyTorch 2.8.x for this notebook")
if str(torch.version.cuda or "").strip() and not str(torch.version.cuda).startswith("12.8"):
    print("⚠️ Recommended: CUDA 12.8 runtime")

print("\nVRAM / batch-size guidance (single GPU):")
print("- H100 SXM 80GB: img=384, bs=64, accum=6 (effective=384)")
print("- A100 80GB:    img=384, bs=32, accum=6 (effective=192)")
print("- RTX 4090 24GB:img=320, bs=10, accum=12 (effective=120)")

print("\nResolved config:")
for k, v in asdict(CFG).items():
    print(f"{k}: {v}")
print(f"grad_accum_steps: {CFG.grad_accum_steps}")
print(f"effective_batch_size: {CFG.effective_batch_size}")

print("Cell 1 complete")

In [ ]:
# Cell 2 — Dataset Verification & Fast Extraction
# Expected output: dataset root confirmation and ready status.

seven_zip = ensure_7z()

DEFAULT_DATA_ROOT = Path(os.environ.get("APTOS_DATA_DIR", "/workspace/aptos2019" if os.name != "nt" else str(Path.cwd() / "aptos2019"))).expanduser()
DATA_ROOT = DEFAULT_DATA_ROOT
DATA_ROOT.mkdir(parents=True, exist_ok=True)

print(f"Dataset root: {DATA_ROOT}")

if not dataset_ready(DATA_ROOT):
    # If the dataset was copied as archives, extract them locally.
    local_archives = sorted(DATA_ROOT.glob("*.zip"))
    if local_archives:
        print(f"Found {len(local_archives)} local archive(s); extracting now.")
        extract_archives_fast(DATA_ROOT, seven_zip)

if not dataset_ready(DATA_ROOT):
    candidates = [p for p in DATA_ROOT.glob("**/train.csv") if p.is_file()]
    if candidates:
        candidate_root = candidates[0].parent
        if (candidate_root / "train_images").exists():
            DATA_ROOT = candidate_root

if not dataset_ready(DATA_ROOT):
    raise FileNotFoundError(
        "APTOS dataset not found. Place the extracted dataset in APTOS_DATA_DIR so train.csv and train_images/ are available."
    )

print("✅ Dataset verified and ready")
print(f"Active dataset root: {DATA_ROOT}")
print("Cell 2 complete")

In [ ]:
# Cell 3 — Data Exploration & Splitting
# Expected output: dataset stats, class distribution, leakage-safe fold assignments.

ID_CANDIDATES = ["id_code", "image_id", "image", "filename", "file_name"]
LABEL_CANDIDATES = ["diagnosis", "label", "target", "class"]

if not TRAIN_CSV_PATH.exists():
    raise RuntimeError(f"Missing CSV: {TRAIN_CSV_PATH}")

df = pd.read_csv(TRAIN_CSV_PATH)


def resolve_col(cols, candidates, env_name):
    override = os.environ.get(env_name)
    if override:
        if override in cols:
            return override
        raise RuntimeError(f"{env_name}={override} is not in columns: {list(cols)}")
    for c in candidates:
        if c in cols:
            return c
    raise RuntimeError(f"Could not resolve column from candidates={candidates}; available={list(cols)}")


ID_COL = resolve_col(df.columns, ID_CANDIDATES, "APTOS_ID_COL")
LABEL_COL = resolve_col(df.columns, LABEL_CANDIDATES, "APTOS_LABEL_COL")

df[LABEL_COL] = pd.to_numeric(df[LABEL_COL], errors="coerce")
if df[LABEL_COL].isna().any():
    raise RuntimeError("Label column contains non-numeric values.")
df[LABEL_COL] = df[LABEL_COL].astype(int)


def to_image_path(image_id):
    image_id = str(image_id)
    for ext in [".png", ".jpg", ".jpeg"]:
        p = TRAIN_IMG_DIR_PATH / f"{Path(image_id).stem}{ext}"
        if p.exists():
            return p
    return TRAIN_IMG_DIR_PATH / f"{Path(image_id).stem}.png"


df["image_path"] = df[ID_COL].map(to_image_path)
df["image_exists"] = df["image_path"].map(lambda p: Path(p).exists())
missing_count = int((~df["image_exists"]).sum())
if missing_count > 0:
    print(f"⚠️ Missing images: {missing_count} (showing first 5)")
    print(df.loc[~df["image_exists"], [ID_COL, "image_path"]].head())

# Patient-level proxy grouping from filename stem prefix
df["patient_group"] = df[ID_COL].astype(str).str.replace(r"[^a-zA-Z0-9_\-]", "_", regex=True).str.split(r"[_\-]").str[0]
if df["patient_group"].isna().any() or (df["patient_group"].astype(str).str.len() == 0).any():
    df["patient_group"] = df[ID_COL].astype(str)

# Stratified folds
skf = StratifiedKFold(n_splits=CFG.n_folds, shuffle=True, random_state=CFG.seed)
df["fold"] = -1
for fold_idx, (_, val_idx) in enumerate(skf.split(df, df[LABEL_COL])):
    df.loc[val_idx, "fold"] = fold_idx

if (df["fold"] < 0).any():
    raise RuntimeError("Fold assignment failed for some rows.")

split_csv = RESULTS_DIR / f"splits_{NOTEBOOK_VERSION}.csv"
df.to_csv(split_csv, index=False)

print(f"Rows: {len(df)}")
print(f"ID column: {ID_COL}")
print(f"Label column: {LABEL_COL}")
print("Class distribution:")
print(df[LABEL_COL].value_counts().sort_index())
print("Fold distribution:")
print(df["fold"].value_counts().sort_index())
print(f"Saved split CSV: {split_csv}")
print("Cell 3 complete")

In [ ]:
# Cell 4 — Dataset + Dataloaders
# Expected output: train/val dataset sizes and dataloader readiness.


class APTOSDataset(Dataset):
    def __init__(self, frame, image_col, label_col, transform=None):
        self.frame = frame.reset_index(drop=True)
        self.image_col = image_col
        self.label_col = label_col
        self.transform = transform

    def __len__(self):
        return len(self.frame)

    def __getitem__(self, idx):
        row = self.frame.iloc[idx]
        path = Path(row[self.image_col])
        img = cv2.imread(str(path), cv2.IMREAD_COLOR)
        if img is None:
            raise RuntimeError(f"Failed to read image: {path}")
        img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
        img = Image.fromarray(img)
        if self.transform is not None:
            img = self.transform(img)
        label = int(row[self.label_col])
        return img, label


train_tfms = T.Compose([
    T.RandomResizedCrop((CFG.img_size, CFG.img_size), scale=(0.85, 1.0), ratio=(0.9, 1.1), antialias=True),
    T.RandomHorizontalFlip(p=0.5),
    T.RandomAffine(
        degrees=12,
        translate=(0.03, 0.03),
        scale=(0.92, 1.08),
        interpolation=T.InterpolationMode.BILINEAR,
        fill=0,
    ),
    T.ColorJitter(brightness=0.15, contrast=0.15, saturation=0.12, hue=0.03),
    T.ToTensor(),
    T.Normalize(mean=(0.485, 0.456, 0.406), std=(0.229, 0.224, 0.225)),
])

val_tfms = T.Compose([
    T.Resize((CFG.img_size, CFG.img_size), antialias=True),
    T.ToTensor(),
    T.Normalize(mean=(0.485, 0.456, 0.406), std=(0.229, 0.224, 0.225)),
])

fold_idx = CFG.fold if CFG.fold >= 0 else 0

train_df = df[df["fold"] != fold_idx].copy()
val_df = df[df["fold"] == fold_idx].copy()

if len(train_df) == 0 or len(val_df) == 0:
    raise RuntimeError(f"Invalid fold split: train={len(train_df)}, val={len(val_df)}")

train_ds = APTOSDataset(train_df, image_col="image_path", label_col=LABEL_COL, transform=train_tfms)
val_ds = APTOSDataset(val_df, image_col="image_path", label_col=LABEL_COL, transform=val_tfms)

pin = DEVICE.type == "cuda"
loader_kwargs = {
    "num_workers": CFG.num_workers,
    "pin_memory": pin,
    "persistent_workers": CFG.num_workers > 0,
}
if CFG.num_workers > 0:
    loader_kwargs["prefetch_factor"] = 4

class_counts = train_df[LABEL_COL].value_counts().sort_index()
class_weights = (1.0 / class_counts).astype(float)
class_weights = class_weights / class_weights.mean()
train_sample_weights = train_df[LABEL_COL].map(class_weights).astype(float).to_numpy()
train_sampler = WeightedRandomSampler(
    weights=torch.as_tensor(train_sample_weights, dtype=torch.double),
    num_samples=len(train_sample_weights),
    replacement=True,
)

train_loader = DataLoader(
    train_ds,
    batch_size=CFG.batch_size,
    sampler=train_sampler,
    shuffle=False,
    drop_last=True,
    **loader_kwargs,
)
val_loader = DataLoader(
    val_ds,
    batch_size=CFG.batch_size,
    shuffle=False,
    **loader_kwargs,
)

print(f"Fold: {fold_idx}")
print(f"Train samples: {len(train_ds)}")
print(f"Val samples: {len(val_ds)}")
print("Train class distribution:")
print(class_counts)
print("Normalized class weights:")
print(class_weights.round(3))
print(f"Train batches/epoch: {len(train_loader)}")
print(f"Val batches: {len(val_loader)}")
print("Cell 4 complete")

In [ ]:
# Cell 5 — Model + Loss + Mixup
# Expected output: model initialized, VRAM snapshot, optimizer/scheduler/scaler ready.

NUM_CLASSES = int(df[LABEL_COL].nunique())


class SoftTargetCrossEntropy(nn.Module):
    def __init__(self, smoothing=0.0, class_weights=None):
        super().__init__()
        self.smoothing = float(smoothing)
        if class_weights is not None:
            self.register_buffer("class_weights", class_weights.float())
        else:
            self.class_weights = None

    def forward(self, logits, targets):
        if targets.ndim == 1:
            targets = F.one_hot(targets, num_classes=logits.shape[1]).float()
        targets = targets * (1.0 - self.smoothing) + self.smoothing / logits.shape[1]
        log_probs = F.log_softmax(logits, dim=1)
        loss = -(targets * log_probs).sum(dim=1)
        if self.class_weights is not None:
            sample_weights = (targets * self.class_weights.unsqueeze(0)).sum(dim=1)
            loss = loss * sample_weights
        return loss.mean()


# --- Mixup / CutMix helpers (reused in Cell 6) ---
def _sample_beta(alpha, device):
    if alpha <= 0:
        return 1.0
    dist = torch.distributions.Beta(alpha, alpha)
    return float(dist.sample().to(device))


def apply_mixup_cutmix(x, y, num_classes, mixup_alpha=0.2, cutmix_alpha=1.0, prob=1.0):
    if torch.rand(1).item() > prob:
        return x, F.one_hot(y, num_classes=num_classes).float()

    bs = x.size(0)
    perm = torch.randperm(bs, device=x.device)
    y_onehot = F.one_hot(y, num_classes=num_classes).float()

    use_cutmix = cutmix_alpha > 0 and torch.rand(1).item() < 0.5
    if use_cutmix:
        lam = _sample_beta(cutmix_alpha, x.device)
        h, w = x.shape[-2:]
        cut_rat = (1.0 - lam) ** 0.5
        ch, cw = int(h * cut_rat), int(w * cut_rat)
        cy = torch.randint(0, h, (1,), device=x.device).item()
        cx = torch.randint(0, w, (1,), device=x.device).item()
        y1, y2 = max(cy - ch // 2, 0), min(cy + ch // 2, h)
        x1, x2 = max(cx - cw // 2, 0), min(cx + cw // 2, w)
        x[:, :, y1:y2, x1:x2] = x[perm, :, y1:y2, x1:x2]
        lam = 1.0 - ((x2 - x1) * (y2 - y1) / (h * w))
    else:
        lam = _sample_beta(mixup_alpha, x.device)
        x = x * lam + x[perm] * (1.0 - lam)

    y_mix = y_onehot * lam + y_onehot[perm] * (1.0 - lam)
    return x, y_mix


def create_model(backbone: str, num_classes: int):
    try:
        return timm.create_model(backbone, pretrained=True, num_classes=num_classes)
    except Exception as exc:
        print(f"⚠️ Pretrained weights unavailable for {backbone}: {exc}")
        print("Falling back to `pretrained=False` so the notebook can continue.")
        return timm.create_model(backbone, pretrained=False, num_classes=num_classes)


model = create_model(CFG.backbone, NUM_CLASSES).to(DEVICE)
if DEVICE.type == "cuda":
    model = model.to(memory_format=torch.channels_last)

if CFG.use_torch_compile and hasattr(torch, "compile") and DEVICE.type == "cuda":
    try:
        model = torch.compile(model)
        print("✅ torch.compile enabled")
    except Exception as exc:
        print(f"⚠️ torch.compile failed ({exc}); continuing without compile")

class_weights_tensor = torch.as_tensor(class_weights.sort_index().to_numpy(), dtype=torch.float32, device=DEVICE)
criterion = SoftTargetCrossEntropy(smoothing=CFG.label_smoothing, class_weights=class_weights_tensor)
optimizer = torch.optim.AdamW(model.parameters(), lr=CFG.lr, weight_decay=CFG.weight_decay)
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=max(1, CFG.epochs))

try:
    scaler = torch.amp.GradScaler("cuda", enabled=CFG.amp and DEVICE.type == "cuda")
except Exception:
    scaler = torch.amp.GradScaler(enabled=CFG.amp and DEVICE.type == "cuda")


def print_vram_snapshot(tag="post-model-init"):
    if DEVICE.type != "cuda":
        print("CUDA not available; VRAM snapshot skipped")
        return
    allocated = torch.cuda.memory_allocated() / (1024**3)
    reserved = torch.cuda.memory_reserved() / (1024**3)
    total = torch.cuda.get_device_properties(0).total_memory / (1024**3)
    print(f"[{tag}] allocated={allocated:.2f}GB reserved={reserved:.2f}GB total={total:.2f}GB")


print_vram_snapshot()
print(f"Model backbone: {CFG.backbone}")
print(f"Classes: {NUM_CLASSES}")
print(f"Gradient accumulation steps: {CFG.grad_accum_steps}")
print(f"Effective batch size: {CFG.effective_batch_size}")
print("Cell 5 complete")

In [ ]:
# Cell 6 — Training Loop
# Expected output: epoch-by-epoch train/val metrics, best checkpoint saved.

BEST_CKPT_PATH = CHECKPOINTS_DIR / f"best_{NOTEBOOK_VERSION}.pt"
history = []
best_qwk = -1.0
best_epoch = -1
patience_counter = 0


def _amp_context():
    if hasattr(torch, "amp") and hasattr(torch.amp, "autocast"):
        return torch.amp.autocast("cuda", enabled=CFG.amp and DEVICE.type == "cuda")
    return torch.autocast(device_type="cuda", enabled=CFG.amp and DEVICE.type == "cuda")


def train_one_epoch(model, loader, optimizer, scaler, criterion):
    model.train()
    running_loss = 0.0
    total = 0

    optimizer.zero_grad(set_to_none=True)
    accum_steps = 0

    pbar = tqdm(enumerate(loader), total=len(loader), leave=False)
    for step, (images, labels) in pbar:
        if DEVICE.type == "cuda":
            images = images.to(DEVICE, non_blocking=True, memory_format=torch.channels_last)
        else:
            images = images.to(DEVICE, non_blocking=True)
        labels = labels.to(DEVICE, non_blocking=True)

        with _amp_context():
            images_mix, labels_mix = apply_mixup_cutmix(
                images,
                labels,
                num_classes=NUM_CLASSES,
                mixup_alpha=CFG.mixup_alpha,
                cutmix_alpha=CFG.cutmix_alpha,
                prob=CFG.mix_prob,
            )
            logits = model(images_mix)
            loss = criterion(logits, labels_mix)
            loss = loss / CFG.grad_accum_steps

        scaler.scale(loss).backward()
        accum_steps += 1

        if accum_steps % CFG.grad_accum_steps == 0:
            if CFG.grad_clip > 0:
                scaler.unscale_(optimizer)
                torch.nn.utils.clip_grad_norm_(model.parameters(), CFG.grad_clip)
            scaler.step(optimizer)
            scaler.update()
            optimizer.zero_grad(set_to_none=True)

        running_loss += loss.item() * CFG.grad_accum_steps * labels.size(0)
        total += labels.size(0)
        pbar.set_description(f"train_loss={running_loss/max(1,total):.4f}")

    if accum_steps % CFG.grad_accum_steps != 0:
        if CFG.grad_clip > 0:
            scaler.unscale_(optimizer)
            torch.nn.utils.clip_grad_norm_(model.parameters(), CFG.grad_clip)
        scaler.step(optimizer)
        scaler.update()
        optimizer.zero_grad(set_to_none=True)

    return running_loss / max(1, total)


@torch.inference_mode()
def validate(model, loader):
    model.eval()
    losses = []
    all_labels = []
    all_preds = []
    all_logits = []

    for images, labels in tqdm(loader, total=len(loader), leave=False):
        if DEVICE.type == "cuda":
            images = images.to(DEVICE, non_blocking=True, memory_format=torch.channels_last)
        else:
            images = images.to(DEVICE, non_blocking=True)
        labels = labels.to(DEVICE, non_blocking=True)

        with _amp_context():
            logits = model(images)
            label_oh = torch.nn.functional.one_hot(labels, num_classes=NUM_CLASSES).float()
            loss = criterion(logits, label_oh)

        preds = logits.argmax(dim=1)

        losses.append(loss.item())
        all_labels.append(labels.cpu().numpy())
        all_preds.append(preds.cpu().numpy())
        all_logits.append(logits.float().cpu().numpy())

    y_true = np.concatenate(all_labels)
    y_pred = np.concatenate(all_preds)
    logits_np = np.concatenate(all_logits)

    acc = accuracy_score(y_true, y_pred)
    qwk = cohen_kappa_score(y_true, y_pred, weights="quadratic")
    bal_acc = balanced_accuracy_score(y_true, y_pred)
    macro_f1 = f1_score(y_true, y_pred, average="macro")
    per_class_recall = recall_score(y_true, y_pred, average=None, labels=list(range(NUM_CLASSES)), zero_division=0)

    return {
        "val_loss": float(np.mean(losses)),
        "val_acc": float(acc),
        "val_qwk": float(qwk),
        "val_bal_acc": float(bal_acc),
        "val_macro_f1": float(macro_f1),
        "val_per_class_recall": per_class_recall,
        "y_true": y_true,
        "y_pred": y_pred,
        "logits": logits_np,
    }


for epoch in range(CFG.epochs):
    t0 = time.time()
    train_loss = train_one_epoch(model, train_loader, optimizer, scaler, criterion)
    val_out = validate(model, val_loader)

    scheduler.step()

    row = {
        "epoch": epoch + 1,
        "train_loss": train_loss,
        "val_loss": val_out["val_loss"],
        "val_acc": val_out["val_acc"],
        "val_qwk": val_out["val_qwk"],
        "val_bal_acc": val_out["val_bal_acc"],
        "val_macro_f1": val_out["val_macro_f1"],
        "lr": optimizer.param_groups[0]["lr"],
        "minutes": (time.time() - t0) / 60.0,
    }
    for c, rec in enumerate(val_out["val_per_class_recall"]):
        row[f"val_recall_{c}"] = float(rec)
    history.append(row)

    print(
        f"Epoch {row['epoch']:02d}/{CFG.epochs} | "
        f"train_loss={row['train_loss']:.4f} val_loss={row['val_loss']:.4f} "
        f"val_acc={row['val_acc']:.4f} val_bal_acc={row['val_bal_acc']:.4f} "
        f"val_macro_f1={row['val_macro_f1']:.4f} val_qwk={row['val_qwk']:.4f} "
        f"time={row['minutes']:.2f}m"
    )

    if row["val_qwk"] > best_qwk:
        best_qwk = row["val_qwk"]
        best_epoch = epoch + 1
        patience_counter = 0
        torch.save({
            "model_state": model.state_dict(),
            "epoch": best_epoch,
            "val_qwk": best_qwk,
            "config": asdict(CFG),
            "notebook_version": NOTEBOOK_VERSION,
        }, BEST_CKPT_PATH)
        print(f"✅ Saved best checkpoint -> {BEST_CKPT_PATH}")
    else:
        patience_counter += 1
        if patience_counter >= CFG.patience:
            print(f"Early stopping at epoch {epoch+1} (patience={CFG.patience})")
            break

if not BEST_CKPT_PATH.exists():
    raise RuntimeError("Training completed but no best checkpoint was saved.")

history_df = pd.DataFrame(history)
history_csv = RESULTS_DIR / f"history_{NOTEBOOK_VERSION}.csv"
history_df.to_csv(history_csv, index=False)
print(f"Best epoch: {best_epoch} | Best QWK: {best_qwk:.4f}")
print(f"History saved: {history_csv}")
print("Cell 6 complete")

In [ ]:
# Cell 7 — Temperature Scaling + TTA
# Expected output: calibrated temperature and TTA-ready inference helper.

try:
    ckpt = torch.load(BEST_CKPT_PATH, map_location=DEVICE, weights_only=True)
except TypeError:
    ckpt = torch.load(BEST_CKPT_PATH, map_location=DEVICE)

model.load_state_dict(ckpt["model_state"])
model.eval()


@torch.no_grad()
def collect_val_logits(model, loader):
    logits_list, labels_list = [], []
    for images, labels in loader:
        if DEVICE.type == "cuda":
            images = images.to(DEVICE, non_blocking=True, memory_format=torch.channels_last)
        else:
            images = images.to(DEVICE, non_blocking=True)
        logits = model(images)
        logits_list.append(logits.detach().float().cpu())
        labels_list.append(labels.detach().cpu())
    return torch.cat(logits_list).clone(), torch.cat(labels_list).clone()


val_logits_t, val_labels_t = collect_val_logits(model, val_loader)

# Learn scalar temperature on validation logits
log_temp = torch.nn.Parameter(torch.zeros(1))
opt_t = torch.optim.LBFGS([log_temp], lr=0.05, max_iter=80)


def nll_obj():
    opt_t.zero_grad()
    temp = torch.exp(log_temp).clamp(min=0.5, max=8.0)
    loss = F.cross_entropy(val_logits_t / temp, val_labels_t)
    loss.backward()
    return loss.detach()


for _ in range(3):
    opt_t.step(nll_obj)

temperature = float(torch.exp(log_temp).clamp(min=0.5, max=8.0).item())
print(f"Calibrated temperature: {temperature:.4f}")


@torch.no_grad()
def infer_logits_tta(model, loader, tta=4):
    model.eval()
    probs_list = []
    for images, _ in loader:
        batch_probs = []
        for _ in range(max(1, tta)):
            if DEVICE.type == "cuda":
                images_ = images.to(DEVICE, non_blocking=True, memory_format=torch.channels_last)
            else:
                images_ = images.to(DEVICE, non_blocking=True)
            with _amp_context():
                logits = model(images_) / temperature
            batch_probs.append(torch.softmax(logits, dim=1).detach().float().cpu())
        probs_list.append(torch.stack(batch_probs).mean(dim=0))
    return torch.cat(probs_list)


print("Cell 7 complete")

In [ ]:
# Cell 8 — Evaluation & OOF
# Expected output: OOF metrics (QWK/ACC), confidence intervals, saved OOF CSV.
# NOTE: Run cells sequentially (0 -> 8). This cell depends on `df` from Cell 3 and `val_df` from Cell 4.


def bootstrap_ci(y_true, y_pred, metric_fn, n_boot=250, seed=42):
    rng = np.random.default_rng(seed)
    idx = np.arange(len(y_true))
    vals = []
    for _ in range(n_boot):
        s = rng.choice(idx, size=len(idx), replace=True)
        vals.append(metric_fn(y_true[s], y_pred[s]))
    return float(np.percentile(vals, 2.5)), float(np.percentile(vals, 97.5))


all_probs = []
all_targets = []

model.eval()
for images, labels in tqdm(val_loader, total=len(val_loader)):
    if DEVICE.type == "cuda":
        images = images.to(DEVICE, non_blocking=True, memory_format=torch.channels_last)
    else:
        images = images.to(DEVICE, non_blocking=True)
    probs = predict_with_tta(model, images, tta=CFG.tta, temperature=TEMPERATURE)
    all_probs.append(probs.cpu().numpy())
    all_targets.append(labels.numpy())

all_probs = np.concatenate(all_probs)
y_true = np.concatenate(all_targets)
y_pred = np.argmax(all_probs, axis=1)

qwk = float(cohen_kappa_score(y_true, y_pred, weights="quadratic"))
acc = float(accuracy_score(y_true, y_pred))
bal_acc = float(balanced_accuracy_score(y_true, y_pred))
macro_f1 = float(f1_score(y_true, y_pred, average="macro"))
per_class_recall = recall_score(y_true, y_pred, average=None, labels=list(range(NUM_CLASSES)), zero_division=0)
qwk_ci = bootstrap_ci(y_true, y_pred, lambda a, b: cohen_kappa_score(a, b, weights="quadratic"), n_boot=250, seed=CFG.seed)
acc_ci = bootstrap_ci(y_true, y_pred, accuracy_score, n_boot=250, seed=CFG.seed)
cm = confusion_matrix(y_true, y_pred)

val_index = val_df.index.to_numpy()
val_ids = df.loc[val_index, ID_COL].astype(str).tolist()
val_paths = df.loc[val_index, "image_path"].astype(str).tolist()
oof_df = pd.DataFrame({
    "id": val_ids,
    "image_path": val_paths,
    "y_true": y_true,
    "y_pred": y_pred,
})
for c in range(NUM_CLASSES):
    oof_df[f"prob_{c}"] = all_probs[:, c]
oof_path = RESULTS_DIR / f"oof_{NOTEBOOK_VERSION}_fold{CFG.fold if CFG.fold >= 0 else 0}.csv"
oof_df.to_csv(oof_path, index=False)

METRICS = {
    "val_qwk": qwk,
    "val_acc": acc,
    "val_bal_acc": bal_acc,
    "val_macro_f1": macro_f1,
    "val_per_class_recall": per_class_recall.tolist(),
    "val_qwk_ci": qwk_ci,
    "val_acc_ci": acc_ci,
    "temperature": TEMPERATURE,
    "oof_path": str(oof_path),
}

print(f"OOF QWK: {qwk:.4f} | 95% CI [{qwk_ci[0]:.4f}, {qwk_ci[1]:.4f}]")
print(f"OOF ACC: {acc:.4f} | 95% CI [{acc_ci[0]:.4f}, {acc_ci[1]:.4f}]")
print(f"OOF Balanced ACC: {bal_acc:.4f}")
print(f"OOF Macro F1: {macro_f1:.4f}")
print(f"Per-class recall: {np.round(per_class_recall, 4).tolist()}")
print(f"Saved OOF: {oof_path}")
print("Cell 8 complete")

In [ ]:
# Cell 9 — Visualization & Artifacts
# Expected output: confusion matrix + training curves saved to results/.

# Confusion matrix
fig_cm = plt.figure(figsize=(6, 5))
sns.heatmap(cm, annot=True, fmt="d", cmap="Blues")
plt.title("Validation Confusion Matrix")
plt.xlabel("Predicted")
plt.ylabel("True")
cm_path = RESULTS_DIR / f"confusion_matrix_{NOTEBOOK_VERSION}.png"
fig_cm.tight_layout()
fig_cm.savefig(cm_path, dpi=150)
plt.show()

# Training curves
if len(history_df) > 0:
    fig_hist, ax = plt.subplots(1, 3, figsize=(16, 4))
    ax[0].plot(history_df["epoch"], history_df["train_loss"], label="train_loss")
    ax[0].plot(history_df["epoch"], history_df["val_loss"], label="val_loss")
    ax[0].set_title("Loss")
    ax[0].legend()

    ax[1].plot(history_df["epoch"], history_df["val_qwk"], label="val_qwk")
    ax[1].plot(history_df["epoch"], history_df["val_bal_acc"], label="val_bal_acc")
    ax[1].plot(history_df["epoch"], history_df["val_macro_f1"], label="val_macro_f1")
    ax[1].set_title("Severity-Aware Metrics")
    ax[1].legend()

    for c in range(NUM_CLASSES):
        col = f"val_recall_{c}"
        if col in history_df.columns:
            ax[2].plot(history_df["epoch"], history_df[col], label=f"recall_{c}")
    ax[2].set_title("Per-Class Recall")
    ax[2].legend()

    hist_plot_path = RESULTS_DIR / f"training_curves_{NOTEBOOK_VERSION}.png"
    fig_hist.tight_layout()
    fig_hist.savefig(hist_plot_path, dpi=150)
    plt.show()
else:
    hist_plot_path = None

artifact_manifest = {
    "version": NOTEBOOK_VERSION,
    "checkpoint": str(BEST_CKPT_PATH),
    "history_csv": str(RESULTS_DIR / f"history_{NOTEBOOK_VERSION}.csv"),
    "split_csv": str(RESULTS_DIR / f"splits_{NOTEBOOK_VERSION}.csv"),
    "oof_csv": METRICS["oof_path"],
    "confusion_matrix": str(cm_path),
    "training_curves": str(hist_plot_path) if hist_plot_path else "N/A",
    "val_qwk": METRICS["val_qwk"],
    "val_acc": METRICS["val_acc"],
    "val_bal_acc": METRICS["val_bal_acc"],
    "val_macro_f1": METRICS["val_macro_f1"],
}

manifest_path = ARTIFACTS_DIR / f"artifact_manifest_{NOTEBOOK_VERSION}.json"
with open(manifest_path, "w", encoding="utf-8") as f:
    json.dump(artifact_manifest, f, indent=2)

print(f"Saved confusion matrix: {cm_path}")
print(f"Saved manifest: {manifest_path}")
print("Cell 9 complete")

In [ ]:
# Cell 10 — Final Summary + Reproducibility Checklist
# Expected output: polished run summary, key metrics, and checklist.

summary_md = f"""
## ✅ APTOS-DR-GPU v4 Run Summary

- **Notebook version:** `{NOTEBOOK_VERSION}`
- **Backbone:** `{CFG.backbone}`
- **Image size:** `{CFG.img_size}`
- **Batch / Accum / Effective:** `{CFG.batch_size}` / `{CFG.grad_accum_steps}` / `{CFG.effective_batch_size}`
- **Torch compile:** `{CFG.use_torch_compile}`
- **Temperature:** `{METRICS['temperature']:.4f}`

### Core Metrics
- **OOF QWK:** `{METRICS['val_qwk']:.4f}` (95% CI `{METRICS['val_qwk_ci'][0]:.4f}`–`{METRICS['val_qwk_ci'][1]:.4f}`)
- **OOF ACC:** `{METRICS['val_acc']:.4f}` (95% CI `{METRICS['val_acc_ci'][0]:.4f}`–`{METRICS['val_acc_ci'][1]:.4f}`)
- **OOF Balanced ACC:** `{METRICS['val_bal_acc']:.4f}`
- **OOF Macro F1:** `{METRICS['val_macro_f1']:.4f}`

### Per-Class Recall
{''.join([f'- **Class {i} Recall:** `{r:.4f}`\n' for i, r in enumerate(METRICS['val_per_class_recall'])])}

### Reproducibility Checklist
- [x] Deterministic seeds set (`seed_everything`)
- [x] Versioned artifacts saved under `results/`, `artifacts/`, `checkpoints/`
- [x] OOF file exported (`oof_{NOTEBOOK_VERSION}_*.csv`)
- [x] Dataset path and extraction flow logged
- [x] Class-balanced sampler enabled for severity imbalance
- [x] Balanced accuracy / macro F1 / per-class recall tracked

### Troubleshooting Quick Hits
1. Missing dataset files: place the extracted APTOS dataset under `APTOS_DATA_DIR`.
2. Slow extraction: confirm `7z` is installed and visible in Cell 2 output.
3. OOM: lower `BATCH_SIZE`, raise `EFFECTIVE_BATCH_TARGET`.
4. GPU usage looks low: increase batch size if VRAM allows and confirm CUDA is active.

**Clinical Disclaimer:** Research use only — not for clinical diagnosis.
"""

display(Markdown(summary_md))

print("Notebook completed successfully.")
print("Cell 10 complete")